# 阵地突围：兵种、伤害、经济与胜率平衡分析

目标：把战线模式的最终生效数值整理为可复算基线，检查单位性价比、装甲伤害曲线、经济节奏与 7 关相对难度。

> 说明：胜率是简化的蒙特卡洛强度模型，不是玩家遥测数据；它适合发现异常关卡和参数方向，不替代实机测试。数据来源为 `index.html` 中最终覆盖后的有效配置。

In [ ]:
from __future__ import annotations
import math, random, statistics
from pprint import pprint

SEED = 20260724
random.seed(SEED)


## 1. 最终生效数值

DPS 按 `单发伤害 / 射击间隔` 计算；装甲有效生命按当前伤害公式中的 `1 - armor × 0.48` 换算。BMPT 的双联火箭另行计入反装甲输出。

In [ ]:
infantry = {
 '步枪兵': dict(cost=3, hp=82, range=205, rate=.68, damage=14, capture=1.00),
 '机枪兵': dict(cost=7, hp=112, range=310, rate=.16, damage=12, capture=.72),
 '狙击兵': dict(cost=9, hp=68, range=525, rate=2.4, damage=72, capture=.55),
 '工兵': dict(cost=10, hp=94, range=245, rate=2.45, damage=82, capture=1.65),
 '指挥官': dict(cost=11, hp=96, range=215, rate=.66, damage=14, capture=.85),
}
vehicles = {
 '悍马车': dict(cost=11, hp=260, armor=.18, range=195, rate=.46, damage=17),
 'CM34': dict(cost=24, hp=520, armor=.38, range=265, rate=.30, damage=20),
 'HSTV': dict(cost=34, hp=670, armor=.55, range=345, rate=1.65, damage=98),
 'BMPT': dict(cost=47, hp=1320, armor=.88, range=315, rate=.30, damage=40, rocket_damage=82, rocket_rate=2.45),
 'M1': dict(cost=54, hp=1160, armor=.79, range=455, rate=2.00, damage=220),
 'T-90M': dict(cost=56, hp=1320, armor=.88, range=415, rate=2.85, damage=184),
 '阿帕奇': dict(cost=66, hp=760, armor=.42, range=390, rate=.16, damage=18),
 '防空悍马': dict(cost=27, hp=310, armor=.22, range=315, rate=.42, damage=11),
}

def table(rows, columns):
    widths = [max(len(str(c)), *(len(str(r.get(c, ''))) for r in rows)) for c in columns]
    print(' | '.join(str(c).ljust(w) for c, w in zip(columns, widths)))
    print('-+-'.join('-' * w for w in widths))
    for r in rows: print(' | '.join(str(r.get(c, '')).ljust(w) for c, w in zip(columns, widths)))

inf_rows=[]
for name,d in infantry.items():
    inf_rows.append({'单位':name,'价格':d['cost'],'生命':d['hp'],'射程':d['range'],'DPS':round(d['damage']/d['rate'],1),'DPS/点':round(d['damage']/d['rate']/d['cost'],1),'占领/点':round(d['capture']/d['cost'],3)})
table(inf_rows, ['单位','价格','生命','射程','DPS','DPS/点','占领/点'])


## 2. 反装甲伤害与生存曲线

轻武器对载具的倍率：步枪 0.075、机枪 0.055、狙击 0.16、工兵 1.35、载具 1.0。曲线用于确认“步兵子弹压制轻装甲、反坦克武器处理重装甲”的设计是否成立。

In [ ]:
anti_armor = {'步枪兵':.075, '机枪兵':.055, '狙击兵':.16, '工兵':1.35, '装甲主武器':1.0}
curve=[]
for target,td in vehicles.items():
    reduction = 1 - td['armor']*.48
    effective_hp = td['hp']/reduction
    curve.append({'目标':target,'减伤%':round((1-reduction)*100,1),'等效生命':round(effective_hp),'工兵击杀秒':round(effective_hp/(82/2.45*1.35),1),'M1击杀秒':round(effective_hp/(220/2),1)})
table(curve, ['目标','减伤%','等效生命','工兵击杀秒','M1击杀秒'])

print('\n攻击者对 T-90M 的实际 DPS：')
t90 = vehicles['T-90M']; reduction = 1-t90['armor']*.48
for name,mult in anti_armor.items():
    if name in infantry: base=infantry[name]['damage']/infantry[name]['rate']
    elif name=='装甲主武器': base=vehicles['M1']['damage']/vehicles['M1']['rate']
    else: continue
    print(f'{name:8s} {base*mult*reduction:6.2f} DPS')


## 3. 经济效率

综合指数只用于横向筛查：`(DPS × 射程系数 + 有效生命 × 生存权重) / 价格`。它不是直接战斗结论；射程、目标选择、溅射、载员投放与防空职责仍需实机评价。

In [ ]:
eff=[]
for name,d in vehicles.items():
    dps=d['damage']/d['rate']
    if name=='BMPT': dps += 2*d['rocket_damage']*1.35/d['rocket_rate']
    ehp=d['hp']/(1-d['armor']*.48)
    score=(dps*(.6+d['range']/500)+ehp*.055)/d['cost']
    eff.append({'载具':name,'价格':d['cost'],'DPS':round(dps,1),'等效生命':round(ehp),'综合/点':round(score,2)})
eff.sort(key=lambda r:r['综合/点'], reverse=True)
table(eff,['载具','价格','DPS','等效生命','综合/点'])

income=1.62
print('\n第三关基础收入下的攒点时间：')
for name in ['悍马车','CM34','HSTV','BMPT','M1','T-90M','阿帕奇']:
    print(f'{name:8s}: {vehicles[name]["cost"]/income:5.1f} 秒')


## 4. 七关相对胜率模拟

模型把收入、敌军生命/伤害倍率、装甲预算、开局阵地差和总部耐久压缩为总战力，并加入玩家决策波动。输出用于找难度突刺，目标区间为前六关约 45%–72%，第七关约 30%–45%。

In [ ]:
levels = [
 ('1 步兵前线',1.45,1.30,.92,.95,0,1800,2,2),
 ('2 机动步兵',1.55,1.40,.96,.98,.18,2000,2,2),
 ('3 装甲输送',1.62,1.52,1,1,.32,2200,2,2),
 ('4 轻坦突破',1.68,1.62,1.03,1.04,.48,2450,2,2),
 ('5 重装甲决战',1.75,1.72,1.06,1.08,.68,2750,2,3),
 ('6 联合作战',1.82,1.80,1.10,1.12,.82,3100,2,3),
 ('7 钢铁洪流',1.85,2.12,1.18,1.22,1.05,3500,2,4),
]

def estimate(level, trials=12000):
    name,pi,ei,ed,eh,armor,hq,ours,theirs=level
    wins=0
    for _ in range(trials):
        player_skill=random.lognormvariate(0,.16)
        enemy_plan=random.lognormvariate(0,.13)
        player=pi*player_skill*(1+.055*ours)
        enemy=ei*ed*eh*enemy_plan*(1+.11*armor)*(1+.045*(theirs-ours))*(hq/2200)**.18
        wins += player>enemy
    return wins/trials

rates=[{'关卡':lv[0],'估算胜率':f'{estimate(lv):.1%}'} for lv in levels]
table(rates,['关卡','估算胜率'])


## 5. 自动结论

以下规则会标记明显离群项，方便每次数值调整后重新运行。

In [ ]:
findings=[]
best=max(eff,key=lambda r:r['综合/点']); worst=min(eff,key=lambda r:r['综合/点'])
if best['综合/点']>worst['综合/点']*2.4: findings.append(f'载具性价比跨度偏大：{best["载具"]} / {worst["载具"]} = {best["综合/点"]/worst["综合/点"]:.1f}×')
mg=next(r for r in inf_rows if r['单位']=='机枪兵'); rifle=next(r for r in inf_rows if r['单位']=='步枪兵')
if mg['DPS/点']>rifle['DPS/点']*1.8: findings.append('机枪兵正面 DPS/点显著高于步枪兵；需依靠架枪、压制衰减和狙击克制限制泛用性。')
bmpt=next(r for r in eff if r['载具']=='BMPT')
if bmpt['综合/点']==best['综合/点']: findings.append('BMPT 在计入双联火箭后位于综合效率首位；建议重点观察其出场率与对主战坦克胜率。')
findings += [
 '指挥官的 18% 减伤与 16% 输出光环不应叠加；应在代码层限制为单个最高光环。',
 '第七关经济、生命、伤害与开局阵地同时提升，属于多重乘算难度；实测若胜率低于 30%，优先削弱敌方经济而非单位伤害。',
 '建议记录实际通关时间、召唤构成、单位伤害与失败波次，用真实数据替换本笔记本的简化胜率模型。'
]
for i,f in enumerate(findings,1): print(f'{i}. {f}')


## 后续验证

- 每次改动 `frontlineDefs` 或 `frontlineVehicleDefs` 后同步本表并重新运行全部单元格。
- 用至少 30 局真实对局记录校准“胜率模拟”的玩家决策波动。
- 单独建立空旷地、战壕内、炮塔战和载具混战四种测试场景，避免综合指数掩盖克制关系。